<a href="https://colab.research.google.com/github/zachdaube/CS485/blob/main/Homework_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS485 & CS584 - Homework 5**



Understanding the decisions made by Graph Neural Networks (GNNs) is crucial for trust, transparency, and model debugging, especially in applications like drug discovery, fraud detection, and social network analysis. GNN explainability helps identify which nodes, edges, or features contribute most to predictions, providing insights into model behavior and potential biases.  

In this Colab we will experiment on scaling up GNNs using [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/notes/introduction.html), [DeepSNAP](https://snap.stanford.edu/deepsnap/) and [NetworkX](https://networkx.org/).

Now let's get started! This Colab should take 1-2 hours to complete.

**Note**: Make sure to **restart and run all** before submission, so that the intermediate variables / packages will carry over to the next cell.

**You must compiling all code cells, including training and visualization, to receive full credits.**

### **Setup**

**Note**: You might need to use GPU for this Colab.

In [ ]:
import torch
import os
print("PyTorch has version {}".format(torch.__version__))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')

In [ ]:
torch_version = str(torch.__version__)
pyg_url = f"https://data.pyg.org/whl/torch-{torch_version}.html"

!pip install ogb
!pip install git+https://github.com/snap-stanford/deepsnap.git
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f {pyg_url}
!pip install torch-geometric pytorch-lightning captum

!pip uninstall networkx -y
!pip uninstall python-louvain -y
!pip uninstall community -y

!pip install python-louvain
!pip install networkx

We will use the following plotting function to visualize the evaluation results. Please make sure to import this function before proceeding this Colab.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
import io
from IPython.display import clear_output, display
import matplotlib
matplotlib.use('Agg')

def create_training_plot():
    plt.figure(figsize=(10, 6))
    plt.ion()

    epochs = []
    train_accs = []
    valid_accs = []
    test_accs = []

    train_line, = plt.plot([], [], 'b-', label='Training')
    valid_line, = plt.plot([], [], 'g-', label='Validation')
    test_line, = plt.plot([], [], 'r-', label='Test')

    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('GNN Mini-Batch Training Progress')
    plt.grid(True)
    plt.legend(loc='lower right')

    return {
        'fig': plt.gcf(),
        'epochs': epochs,
        'train_accs': train_accs,
        'valid_accs': valid_accs,
        'test_accs': test_accs,
        'train_line': train_line,
        'valid_line': valid_line,
        'test_line': test_line
    }

def update_plot(plot_data, epoch, train_acc, valid_acc, test_acc):
    plot_data['epochs'].append(epoch)
    plot_data['train_accs'].append(train_acc * 100)
    plot_data['valid_accs'].append(valid_acc * 100)
    plot_data['test_accs'].append(test_acc * 100)

    # Update line data
    plot_data['train_line'].set_data(plot_data['epochs'], plot_data['train_accs'])
    plot_data['valid_line'].set_data(plot_data['epochs'], plot_data['valid_accs'])
    plot_data['test_line'].set_data(plot_data['epochs'], plot_data['test_accs'])

    # Adjust limits if needed
    plt.xlim(0, max(plot_data['epochs']) + 1)
    plt.ylim(min(min(plot_data['train_accs']), min(plot_data['valid_accs']),
                min(plot_data['test_accs'])) - 5,
             max(max(plot_data['train_accs']), max(plot_data['valid_accs']),
                max(plot_data['test_accs'])) + 5)

    clear_output(wait=True)
    display(plot_data['fig'])
    plt.pause(0.1)

# Mark the best model on the plot
def mark_best_model(plot_data, best_epoch, best_train_acc, best_valid_acc, best_test_acc):
    plt.axvline(x=best_epoch, color='r', linestyle='--',
                label=f'Best Model (Epoch {best_epoch})')

    plt.plot(best_epoch, best_train_acc * 100, 'bo', markersize=8)
    plt.plot(best_epoch, best_valid_acc * 100, 'go', markersize=8)
    plt.plot(best_epoch, best_test_acc * 100, 'ro', markersize=8)

    plt.legend(loc='lower right')

    # Display final plot
    clear_output(wait=True)
    display(plot_data['fig'])

### **1 PyTorch Geometric Neighbor Sampling**

The Neighbor Sampling method, originally proposed in [GraphSAGE](https://arxiv.org/abs/1706.02216), is a widely used approach to improve the scalability of GNNs. Instead of aggregating information from all neighbors of a node, which can be computationally expensive in large graphs, GraphSAGE samples a fixed-size subset of neighbors at each layer during training.

Mathematically, at layer $k$, the hidden representation of a node $v$ is updated as:

$$
h_v^{(k)} = \sigma \left( W_k \cdot \text{CONCAT} \left( h_v^{(k-1)}, \text{AGGREGATE}_k \left( \{ h_u^{(k-1)} \mid u \in \mathcal{N}_k(v) \} \right) \right) \right)
$$

where $\mathcal{N}_k(v)$ is the sampled neighborhood of $v$ at depth $k$, $\text{AGGREGATE}_k(\cdot)$ is a differentiable function (e.g., mean, pooling, or LSTM-based), $W_k$ is a trainable weight matrix, and $\sigma$ is a non-linear activation function.

This sampling strategy ensures that only a subset of the neighborhood is loaded into memory at each step, making it feasible to train GNNs on large-scale graphs.

#### **Load data with NeighborSampler**

PyTorch Geometric has implemented the Neighbor Sampling method as the [NeighborSampler](https://pytorch-geometric.readthedocs.io/en/latest/modules/data.html#torch_geometric.data.NeighborSampler) in `torch_geometric.data`. If you are interested in memory-efficient aggregations, please refer to PyG's [Memory-Efficient Aggregations](https://pytorch-geometric.readthedocs.io/en/latest/notes/sparse_tensor.html).

In this cell, the `NeighborSampler` is used when loading data to enable efficient mini-batch training by sampling a fixed number of neighbors instead of using the entire graph. This reduces memory usage and computational cost, making it feasible to train GNNs on large-scale datasets.

In [ ]:
import torch_geometric.transforms as T
from torch_geometric.data import NeighborSampler
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator


dataset_name = 'ogbn-arxiv'
dataset = PygNodePropPredDataset(name=dataset_name, transform=T.ToSparseTensor())
data = dataset[0].to(device)
data.adj_t = data.adj_t.to_symmetric()

split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)

# Sample 10 neighbors for each node in the first layer and 5 for the second layer
train_loader = NeighborSampler(data.adj_t, node_idx=train_idx,
                 sizes=[10, 5], batch_size=4096,
                 shuffle=True, num_workers=2)

# Specify size as -1 to include all neighbors
all_loader = NeighborSampler(data.adj_t, node_idx=None, sizes=[-1],
                batch_size=4096, shuffle=False,
                num_workers=2)
evaluator = Evaluator(name='ogbn-arxiv')

#### **Question 1: GraphSAGE Mini-Batch Implementation. (20 points)**

After loading data with the `NeighborSampler`, we also need to modify the GNN model to let it support the **mini-batch training**.

You'll implement mini-batch training for a GraphSAGE model using PyTorch Geometric. Your task is to complete the implementation of:
1. The forward method in the `SAGE` class to handle batch processing;
2. The `train` function to properly process mini-batches during training.

The `forward` function will take the node feature `x` and a list of three-element tuples `adjs`. Each element in `adjs` contains following elements:
* `edge_index`: The edge index tensor between source and destination nodes, which forms a bipartite grpah.
* `e_id`: The indices of the edges in the original graph.
* `size`: The shape of the bipartite graph, in (*number of source nodes*, *number of destination nodes*) format.

**Note: You must compiling all code cells, including training and visualization, to receive full credits.**

In [ ]:
import torch
import torch.nn.functional as F
import copy
from torch_geometric.nn import SAGEConv


class SAGE(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout):
        super(SAGE, self).__init__()

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()

        self.convs.append(SAGEConv(input_dim, hidden_dim))
        self.bns.append(torch.nn.BatchNorm1d(hidden_dim))

        for i in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(torch.nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, output_dim))

        self.softmax = torch.nn.LogSoftmax(dim=1)
        self.dropout = dropout
        self.num_layers = num_layers

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for bn in self.bns:
            bn.reset_parameters()

    def forward(self, x, adjs, mode="batch"):
        if mode == "batch":

            ############# Your code here #############

            # For each GNN layer i:
            #   a) Extract target node features using the size information
            #   b) Apply the graph convolution with source and target features
            #   c) For all layers except the last one, apply normalization and activation

            #########################################
        else:
            # Full-batch mode
            for i, conv in enumerate(self.convs):
                x = conv(x, adjs)
                if i != self.num_layers - 1:
                    x = self.bns[i](x)
                    x = F.relu(x)
                    x = F.dropout(x, p=self.dropout, training=self.training)
        return self.softmax(x)

    def inference(self, x_all, all_loader):
        for i in range(self.num_layers):
            xs = []
            for batch_size, n_id, adj in all_loader:
                edge_index, _, size = adj.to(device)
                x = x_all[n_id].to(device)
                x_target = x[:size[1]]
                x = self.convs[i]((x, x_target), edge_index)
                if i != self.num_layers - 1:
                    x = self.bns[i](x)
                    x = F.relu(x)
                    x = F.dropout(x, p=self.dropout, training=self.training)
                xs.append(x.cpu())

            x_all = torch.cat(xs, dim=0)

        return x_all

def train(model, data, train_loader, train_idx, optimizer, loss_fn, mode="batch"):
    model.train()

    total_loss = 0
    if mode == "batch":
        ############# Your code here #############

        # Hint: You can refer to the implementation of full-batch version.
        # For all batches in train_loader, you should:
        #   a) Get node embeddings by indexing features for required nodes
        #   b) Extract labels only for target nodes (first batch_size nodes)
        #   c) Compute loss between predictions and labels. Then backward pass to compute gradients

        #########################################
    else:
        # Full-batch training mode
        optimizer.zero_grad()
        out = model(data.x, data.adj_t, mode=mode)[train_idx]
        train_label = data.y.squeeze(1)[train_idx]
        loss = loss_fn(out, train_label)
        loss.backward()
        optimizer.step()
        total_loss = loss.item()

    return total_loss

@torch.no_grad()
def test(model, data, all_loader, split_idx, evaluator, mode="batch"):
    """
    Evaluate the model on validation and test sets.
    """
    model.eval()

    if mode == "batch":
        out = model.inference(data.x, all_loader)
    else:
        out = model(data.x, data.adj_t, mode="all")

    y_true = data.y.cpu()
    y_pred = out.argmax(dim=-1, keepdim=True)

    train_acc = evaluator.eval({
        'y_true': y_true[split_idx['train']],
        'y_pred': y_pred[split_idx['train']],
    })['acc']
    valid_acc = evaluator.eval({
        'y_true': y_true[split_idx['valid']],
        'y_pred': y_pred[split_idx['valid']],
    })['acc']
    test_acc = evaluator.eval({
        'y_true': y_true[split_idx['test']],
        'y_pred': y_pred[split_idx['test']],
    })['acc']

    return train_acc, valid_acc, test_acc


##### **Mini-Batch Training**

After we implement the `SAGE` model, we can proceed with training and visualize both the training and test performance. First, we evaluate the performance of Mini-Batch Training.

**Note**: No need to modify the parameters.

In [ ]:
args = {
    'device': device,
    'num_layers': 2,
    'hidden_dim': 128,
    'dropout': 0.5,
    'lr': 0.01,
    'epochs': 100,
}

batch_model = SAGE(data.num_features, args['hidden_dim'],
            dataset.num_classes, args['num_layers'],
            args['dropout']).to(device)
batch_model.reset_parameters()

optimizer = torch.optim.Adam(batch_model.parameters(), lr=args['lr'])
loss_fn = F.nll_loss

best_batch_model = None
best_valid_acc = 0
best_epoch = 0
best_result = None

batch_results = []

# Initialize the plot
plot_data = create_training_plot()

for epoch in range(1, 1 + args["epochs"]):
    loss = train(batch_model, data, train_loader, train_idx, optimizer, loss_fn, mode="batch")
    result = test(batch_model, data, all_loader, split_idx, evaluator, mode="batch")
    batch_results.append(result)
    train_acc, valid_acc, test_acc = result

    # Update the plot with new data
    update_plot(plot_data, epoch, train_acc, valid_acc, test_acc)

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_batch_model = copy.deepcopy(batch_model)
        best_epoch = epoch
        best_result = result

    print(f'Epoch: {epoch:02d}, '
          f'Loss: {loss:.4f}, '
          f'Train: {100 * train_acc:.2f}%, '
          f'Valid: {100 * valid_acc:.2f}% '
          f'Test: {100 * test_acc:.2f}%')

# Get final best result
final_result = test(best_batch_model, data, all_loader, split_idx, evaluator, mode="batch")
train_acc, valid_acc, test_acc = final_result

# Mark the best model on the plot
mark_best_model(plot_data, best_epoch, train_acc, valid_acc, test_acc)

print(f'Best model (Epoch {best_epoch}): '
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * valid_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')

##### **Full-Batch Training**

Then, we evaluate the performance of Full-Batch Training with the same parameters.

In [ ]:
all_model = SAGE(data.num_features, args['hidden_dim'],
            dataset.num_classes, args['num_layers'],
            args['dropout']).to(device)
all_model.reset_parameters()

optimizer = torch.optim.Adam(all_model.parameters(), lr=args['lr'])
loss_fn = F.nll_loss

best_all_model = None
best_valid_acc = 0
best_epoch = 0
best_result = None

all_results = []

plot_data = create_training_plot()

for epoch in range(1, 1 + args["epochs"]):
    loss = train(all_model, data, train_loader, train_idx, optimizer, loss_fn, mode="all")
    result = test(all_model, data, all_loader, split_idx, evaluator, mode="all")
    all_results.append(result)
    train_acc, valid_acc, test_acc = result

    update_plot(plot_data, epoch, train_acc, valid_acc, test_acc)

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_all_model = copy.deepcopy(all_model)
        best_epoch = epoch
        best_result = result

    print(f'Epoch: {epoch:02d}, '
          f'Loss: {loss:.4f}, '
          f'Train: {100 * train_acc:.2f}%, '
          f'Valid: {100 * valid_acc:.2f}% '
          f'Test: {100 * test_acc:.2f}%')

# Get final result
final_result = test(best_all_model, data, all_loader, split_idx, evaluator, mode="all")
train_acc, valid_acc, test_acc = final_result
mark_best_model(plot_data, best_epoch, train_acc, valid_acc, test_acc)

print(f'Best model (Epoch {best_epoch}): '
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * valid_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')


### **2 Neighbor Sampling with Different Ratios**

Now, let us customize a simplified version of Neighbor Sampling by using [DeepSNAP](https://snap.stanford.edu/deepsnap/) and [NetworkX](https://networkx.org/), and train models with different neighborhood sampling ratios. We then can analyze the impact of different sampling ratios.

#### **Question 2: Graph Neural Network with Customized Neighbor Sampling. (15 points)**

We'll implement neighbor sampling for Graph Neural Networks (GNNs). Neighbor sampling is a crucial technique for scaling GNNs to large graphs by reducing memory requirements during training. Instead of using the entire graph for each forward pass, we sample a subgraph around our target nodes.

Your task is to complete the implementation of:
1. The sample_neighbors function to perform random neighbor sampling (`sample_neighbors` function);
2. The neighbor_sampling function to build a multi-layer sampling structure (`neighbor_sampling` function).

**Note: You must compiling all code cells, including training and visualization, to receive full credits.**

In [ ]:
import torch
import torch.nn.functional as F
import random
import networkx as nx
import copy
import numpy as np
import torch.nn as nn

from torch_geometric.nn import SAGEConv
from torch.utils.data import DataLoader
from torch.nn import Sequential, Linear, ReLU
from deepsnap.dataset import GraphDataset
from deepsnap.graph import Graph


class GNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, args):
        super(GNN, self).__init__()
        self.dropout = args['dropout']
        self.num_layers = args['num_layers']

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        self.convs.append(SAGEConv(input_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))

        for l in range(self.num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, hidden_dim))

        self.post_mp = nn.Linear(hidden_dim, output_dim)

    def forward(self, data, mode="batch"):
        if mode == "batch":
            edge_indices, x = data
            for i in range(len(self.convs) - 1):
                edge_index = edge_indices[i]
                x = self.convs[i](x, edge_index)
                x = self.bns[i](x)
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
            x = self.convs[-1](x, edge_indices[len(self.convs) - 1])
        else:
            x, edge_index = data.node_feature, data.edge_index
            for i in range(len(self.convs) - 1):
                x = self.convs[i](x, edge_index)
                x = self.bns[i](x)
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
            x = self.convs[-1](x, edge_index)
        x = self.post_mp(x)
        x = F.log_softmax(x, dim=1)
        return x

def nodes_to_tensor(nodes):
    # Transform a set of nodes to node index tensor
    node_label_index = torch.tensor(list(nodes), dtype=torch.long)
    return node_label_index

def edges_to_tensor(edges):
    # Transform a set of edges to edge index tensor
    edge_index = torch.tensor(list(edges), dtype=torch.long)
    edge_index = torch.cat([edge_index, torch.flip(edge_index, [1])], dim=0)
    edge_index = edge_index.permute(1, 0)
    return edge_index

def relable(nodes, labeled_nodes, edges_list):
    # Relabel nodes, labeled_nodes and edges_list with consecutive indices
    relabled_edges_list = []
    sorted_nodes = sorted(nodes)
    node_mapping = {node : i for i, node in enumerate(sorted_nodes)}
    for orig_edges in edges_list:
        relabeled_edges = []
        for edge in orig_edges:
            relabeled_edges.append((node_mapping[edge[0]], node_mapping[edge[1]]))
        relabled_edges_list.append(relabeled_edges)
    relabeled_labeled_nodes = [node_mapping[node] for node in labeled_nodes]
    relabeled_nodes = [node_mapping[node] for node in nodes]
    return relabled_edges_list, relabeled_nodes, relabeled_labeled_nodes

def sample_neighbors(nodes, G, ratio, all_nodes):

    neighbors = set()
    edges = []

    ############# Your code here #############

    # Your code should be ~8-10 lines
    # For each node in the input nodes set:
    #    a) Get its neighbors from the graph G using nx.neighbors()
    #    b) Calculate how many neighbors to sample using the ratio
    #    c) Randomly shuffle and select that many neighbors
    #    d) Add selected neighbors to the neighbors set
    #    e) Create edges between neighbors and the original node

    #########################################

    return neighbors, neighbors.union(all_nodes), edges

def neighbor_sampling(graph, K=2, ratios=(0.1, 0.1, 0.1)):

    assert K + 1 == len(ratios)

    # Extract labeled nodes and apply sampling ratio
    labeled_nodes = graph.node_label_index.tolist()
    random.shuffle(labeled_nodes)
    num = int(len(labeled_nodes) * ratios[-1])
    if num > 0:
        labeled_nodes = labeled_nodes[:num]

    # Initialize lists for nodes and edges
    nodes_list = [set(labeled_nodes)]
    edges_list = []
    all_nodes = labeled_nodes

    ############# Your code here #############

    # For each layer (total of K layers):
    #    a) Get the appropriate sampling ratio for this layer
    #    b) Sample neighbors for the nodes from the previous layer
    #    c) Add the new nodes and edges to our lists

    #########################################

    nodes_list.reverse()
    edges_list.reverse()

    # Process the collected nodes and edges
    relabled_edges_list, relabeled_all_nodes, relabeled_labeled_nodes = \
        relable(all_nodes, labeled_nodes, edges_list)

    # Convert to tensor format
    node_index = nodes_to_tensor(relabeled_all_nodes)
    node_feature = graph.node_feature[node_index]
    edge_indices = [edges_to_tensor(edges) for edges in relabled_edges_list]
    node_label_index = nodes_to_tensor(relabeled_labeled_nodes)

    print(f"Sampled {node_feature.shape[0]} nodes, {edge_indices[0].shape[1] // 2} edges, {node_label_index.shape[0]} labeled nodes")

    return node_feature, edge_indices, node_label_index

def train(train_graphs, val_graphs, args, model, optimizer, mode="batch"):
    best_val = 0
    best_model = None
    accs = []
    graph_train = train_graphs[0]
    graph_train.to(args['device'])
    for epoch in range(1, 1 + args['epochs']):
        model.train()
        optimizer.zero_grad()
        if mode == "batch":
            node_feature, edge_indices, node_label_index = neighbor_sampling(graph_train, args['num_layers'], args['ratios'])
            node_feature = node_feature.to(args['device'])
            node_label_index = node_label_index.to(args['device'])
            for i in range(len(edge_indices)):
                edge_indices[i] = edge_indices[i].to(args['device'])
            pred = model([edge_indices, node_feature])
            pred = pred[node_label_index]
            label = graph_train.node_label[node_label_index]
        elif mode == "community":
            graph = random.choice(train_graphs)
            graph = graph.to(args['device'])
            pred = model(graph, mode="all")
            pred = pred[graph.node_label_index]
            label = graph.node_label[graph.node_label_index]
        else:
            pred = model(graph_train, mode="all")
            label = graph_train.node_label
            pred = pred[graph_train.node_label_index]
        loss = F.nll_loss(pred, label)
        loss.backward()
        optimizer.step()

        train_acc, val_acc, test_acc = test(val_graphs, model)
        accs.append((train_acc, val_acc, test_acc))
        if val_acc > best_val:
            best_val = val_acc
            best_model = copy.deepcopy(model)
    return best_model, accs

def test(graphs, model):
    model.eval()
    accs = []
    for graph in graphs:
        graph = graph.to(args['device'])
        pred = model(graph, mode="all")
        label = graph.node_label
        pred = pred[graph.node_label_index].max(1)[1]
        acc = pred.eq(label).sum().item()
        acc /= len(label)
        accs.append(acc)
    return accs


##### **Full-Batch Training**

Now, we can proceed with training and visualize both the training and test performance. First, we evaluate the performance of Full-Batch Training.

**Note**: No need to modify the parameters.

In [ ]:
from torch_geometric.datasets import Planetoid

# Your original code with visualization added
args = {
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'dropout': 0.5,
    'num_layers': 2,
    'hidden_size': 64,
    'lr': 0.005,
    'epochs': 50,
    'ratios': (0.8, 0.8, 1),
}

pyg_dataset = Planetoid('./tmp', "Cora")

graphs_train, graphs_val, graphs_test = \
    GraphDataset.pyg_to_graphs(pyg_dataset, verbose=True, fixed_split=True)

graph_train = graphs_train[0]
graph_val = graphs_val[0]
graph_test = graphs_test[0]

model = GNN(graph_train.num_node_features, args['hidden_size'], graph_train.num_node_labels, args).to(args['device'])
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
graphs = [graph_train, graph_val, graph_test]

plot_data = create_training_plot()
plt.title('GNN Training Progress (all mode)')

best_epoch = None
best_val_acc = 0

# Using the original train function but capturing results for visualization
all_best_model, all_accs = train(graphs, graphs, args, model, optimizer, mode="all")

# Visualize the training progress
for epoch, (train_acc, val_acc, test_acc) in enumerate(all_accs, 1):
    update_plot(plot_data, epoch, train_acc, val_acc, test_acc)

    # Track best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch

# Get final accuracies of the best model
train_acc, val_acc, test_acc = test([graph_train, graph_val, graph_test], all_best_model)
mark_best_model(plot_data, best_epoch, train_acc, val_acc, test_acc)

print('Best model:',
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * val_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')

Then, let us proceed with training and visualize both the training and test performance when using different sampling ratios.

##### **Sampling with Ratios 0.8**



In [ ]:
args['ratios'] = (0.8, 0.8, 1)

graphs_train, graphs_val, graphs_test = \
    GraphDataset.pyg_to_graphs(pyg_dataset, verbose=True, fixed_split=True)

graph_train = graphs_train[0]
graph_val = graphs_val[0]
graph_test = graphs_test[0]

model = GNN(graph_train.num_node_features, args['hidden_size'], graph_train.num_node_labels, args).to(args['device'])
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
graphs = [graph_train, graph_val, graph_test]

plot_data = create_training_plot()
plt.title('GNN Training Progress (ratio=0.8)')

best_epoch = None
best_val_acc = 0

batch_best_model, batch_accs = train(graphs, graphs, args, model, optimizer)
for epoch, (train_acc, val_acc, test_acc) in enumerate(batch_accs, 1):
    update_plot(plot_data, epoch, train_acc, val_acc, test_acc)

    # Track best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch

# Get final accuracies of the best model
train_acc, val_acc, test_acc = test([graph_train, graph_val, graph_test], batch_best_model)
mark_best_model(plot_data, best_epoch, train_acc, val_acc, test_acc)

print('Best model:',
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * val_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')

##### **Sampling with Ratios 0.3**

In [ ]:
# Change the ratio to 0.3
args['ratios'] = (0.3, 0.3, 1)

graphs_train, graphs_val, graphs_test = \
    GraphDataset.pyg_to_graphs(pyg_dataset, verbose=True, fixed_split=True)

graph_train = graphs_train[0]
graph_val = graphs_val[0]
graph_test = graphs_test[0]

model = GNN(graph_train.num_node_features, args['hidden_size'], graph_train.num_node_labels, args).to(args['device'])
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
graphs = [graph_train, graph_val, graph_test]

plot_data = create_training_plot()
plt.title('GNN Training Progress (ratio=0.3)')

best_epoch = None
best_val_acc = 0

batch_best_model, batch_accs_1 = train(graphs, graphs, args, model, optimizer)
for epoch, (train_acc, val_acc, test_acc) in enumerate(batch_accs_1, 1):
    update_plot(plot_data, epoch, train_acc, val_acc, test_acc)

    # Track best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch

train_acc, val_acc, test_acc = test([graph_train, graph_val, graph_test], batch_best_model)
mark_best_model(plot_data, best_epoch, train_acc, val_acc, test_acc)

print('Best model:',
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * val_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')

### **3 Cluster Sampling**

Instead of neighbor sampling, another approach to scaling up GNNs is subgraph (cluster) sampling. This method, introduced in Cluster-GCN ([Chiang et al., 2019](https://arxiv.org/abs/1905.07953)), partitions the graph into clusters and samples entire clusters for training, rather than individual nodes or neighbors.

Mathematically, given a graph $G = (V, E)$ with $N$ nodes, the clustering algorithm partitions $V$ into $k$ disjoint clusters $C_1, C_2, ..., C_k$. A mini-batch then consists of a randomly selected subset of clusters:

$$
\mathcal{B} = \bigcup_{i \in S} C_i, \quad S \subseteq \{1, 2, ..., k\}
$$

where $S$ is a randomly sampled set of cluster indices.

Since nodes within the same cluster are densely connected, this approach maintains neighborhood information while reducing computational cost. The intra-cluster connectivity preserves message passing efficiency, leading to better scalability compared to standard neighbor sampling.

In this section, we will implement vanilla Cluster-GCN and experiment with 3 different community partition algorithms:
* [Kernighan-Lin algorithm (bisection)](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.kernighan_lin.kernighan_lin_bisection.html)
* [Clauset-Newman-Moore greedy modularity maximization](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.modularity_max.greedy_modularity_communities.html#networkx.algorithms.community.modularity_max.greedy_modularity_communities)
* [Louvain algorithm](https://python-louvain.readthedocs.io/en/latest/api.html)

To make the training more stable, we discard the cluster that has less than 10 nodes.



#### **Question 3: Cluster Sampling in GNNs. (15 points)**

You only need to implement the `louvain` algorithm. More details are provided in [Community API](https://python-louvain.readthedocs.io/en/latest/api.html).

After implementing the clustering algorithm, you only need to run the training and visualization.

**Note: You must compiling all code cells, including training and visualization, to receive full credits.**


In [ ]:
import community as community_louvain

def preprocess(G, node_label_index, method="louvain"):
    graphs = []
    labeled_nodes = set(node_label_index.tolist())
    if method == "louvain":
        communities = {}

        ######### You code here ############

        # You can apply the community.best_partition to partition the graph into communities

        # Process each node in the community mapping:
        #   a) If this community already exists in our dictionary, add the node to it
        #   b) Otherwise, create a new community entry with this node

        ########################################

        communities = communities.values()

    elif method == "bisection":
        communities = nx.algorithms.community.kernighan_lin_bisection(G)
    elif method == "greedy":
        communities = nx.algorithms.community.greedy_modularity_communities(G)

    for community in communities:
        nodes = set(community)
        subgraph = G.subgraph(nodes)
        # Make sure each subgraph has more than 10 nodes
        if subgraph.number_of_nodes() > 10:
            node_mapping = {node : i for i, node in enumerate(subgraph.nodes())}
            subgraph = nx.relabel_nodes(subgraph, node_mapping)
            # Get the id of the training set labeled node in the new graph
            train_label_index = []
            for node in labeled_nodes:
                if node in node_mapping:
                    # Append relabeled labeled node index
                    train_label_index.append(node_mapping[node])

            # Make sure the subgraph contains at least one training set labeled node
            if len(train_label_index) > 0:
                dg = Graph(subgraph)
                # Update node_label_index
                dg.node_label_index = torch.tensor(train_label_index, dtype=torch.long)
                graphs.append(dg)
    return graphs

##### **Louvain Training**

Now, we can proceed with training and visualize both the training and test performance. We first evaluate the performance of Louvain Training.

In [ ]:
args = {
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'dropout': 0.5,
    'num_layers': 2,
    'hidden_size': 64,
    'lr': 0.005,
    'epochs': 150,
}

graphs_train, graphs_val, graphs_test = \
    GraphDataset.pyg_to_graphs(pyg_dataset, verbose=True, fixed_split=True)

graph_train = graphs_train[0]
graph_val = graphs_val[0]
graph_test = graphs_test[0]
graphs = preprocess(graph_train.G, graph_train.node_label_index, method="louvain")
print("Partition the graph in to {} communities".format(len(graphs)))
avg_num_nodes = 0
avg_num_edges = 0
for graph in graphs:
    avg_num_nodes += graph.num_nodes
    avg_num_edges += graph.num_edges
avg_num_nodes = int(avg_num_nodes / len(graphs))
avg_num_edges = int(avg_num_edges / len(graphs))
print("Each community has {} nodes in average".format(avg_num_nodes))
print("Each community has {} edges in average".format(avg_num_edges))

plot_data = create_training_plot()
plt.title('GNN Training Progress (Louvain Community Mode)')

best_epoch = None
best_val_acc = 0

model = GNN(graph_train.num_node_features, args['hidden_size'], graph_train.num_node_labels, args).to(args['device'])
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])

louvain_best_model, louvain_accs = train(graphs, [graph_train, graph_val, graph_test], args, model, optimizer, mode="community")
for epoch, (train_acc, val_acc, test_acc) in enumerate(louvain_accs, 1):
    update_plot(plot_data, epoch, train_acc, val_acc, test_acc)

    # Track best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch

# Get final accuracies of the best model
train_acc, val_acc, test_acc = test([graph_train, graph_val, graph_test], louvain_best_model)
mark_best_model(plot_data, best_epoch, train_acc, val_acc, test_acc)

print('Best model:',
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * val_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')

##### **Bisection Training**

In [ ]:
graphs_train, graphs_val, graphs_test = \
    GraphDataset.pyg_to_graphs(pyg_dataset, verbose=True, fixed_split=True)

graph_train = graphs_train[0]
graph_val = graphs_val[0]
graph_test = graphs_test[0]
graphs = preprocess(graph_train.G, graph_train.node_label_index, method="bisection")
print("Partition the graph in to {} communities".format(len(graphs)))
avg_num_nodes = 0
avg_num_edges = 0
for graph in graphs:
    avg_num_nodes += graph.num_nodes
    avg_num_edges += graph.num_edges
avg_num_nodes = int(avg_num_nodes / len(graphs))
avg_num_edges = int(avg_num_edges / len(graphs))
print("Each community has {} nodes in average".format(avg_num_nodes))
print("Each community has {} edges in average".format(avg_num_edges))

plot_data = create_training_plot()
plt.title('GNN Training Progress (Bisection Community Mode)')

best_epoch = None
best_val_acc = 0

model = GNN(graph_train.num_node_features, args['hidden_size'], graph_train.num_node_labels, args).to(args['device'])
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])

bisection_best_model, bisection_accs = train(graphs, [graph_train, graph_val, graph_test], args, model, optimizer, mode="community")
for epoch, (train_acc, val_acc, test_acc) in enumerate(bisection_accs, 1):
    update_plot(plot_data, epoch, train_acc, val_acc, test_acc)

    # Track best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch

# Get final accuracies of the best model
train_acc, val_acc, test_acc = test([graph_train, graph_val, graph_test], bisection_best_model)
mark_best_model(plot_data, best_epoch, train_acc, val_acc, test_acc)

print('Best model:',
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * val_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')

##### **Greedy Training**

In [ ]:
graphs_train, graphs_val, graphs_test = \
    GraphDataset.pyg_to_graphs(pyg_dataset, verbose=True, fixed_split=True)

graph_train = graphs_train[0]
graph_val = graphs_val[0]
graph_test = graphs_test[0]
graphs = preprocess(graph_train.G, graph_train.node_label_index, method="greedy")
print("Partition the graph in to {} communities".format(len(graphs)))
avg_num_nodes = 0
avg_num_edges = 0
for graph in graphs:
    avg_num_nodes += graph.num_nodes
    avg_num_edges += graph.num_edges
avg_num_nodes = int(avg_num_nodes / len(graphs))
avg_num_edges = int(avg_num_edges / len(graphs))
print("Each community has {} nodes in average".format(avg_num_nodes))
print("Each community has {} edges in average".format(avg_num_edges))

plot_data = create_training_plot()
plt.title('GNN Training Progress (Greedy Community Mode)')

best_epoch = None
best_val_acc = 0

model = GNN(graph_train.num_node_features, args['hidden_size'], graph_train.num_node_labels, args).to(args['device'])
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])

greedy_best_model, greedy_accs = train(graphs, [graph_train, graph_val, graph_test], args, model, optimizer, mode="community")
for epoch, (train_acc, val_acc, test_acc) in enumerate(greedy_accs, 1):
    update_plot(plot_data, epoch, train_acc, val_acc, test_acc)

    # Track best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch

# Get final accuracies of the best model
train_acc, val_acc, test_acc = test([graph_train, graph_val, graph_test], greedy_best_model)
mark_best_model(plot_data, best_epoch, train_acc, val_acc, test_acc)

print('Best model:',
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * val_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')